# simple_mcp_client.py
import asyncio
import subprocess
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain.agents import Tool

async def load_mcp_tools(server_script_path: str):
    """Load tools from an MCP server"""
    tools = []
    
    try:
        server_params = StdioServerParameters(
            command=sys.executable,
            args=[server_script_path]
        )
        
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                
                # Get available tools
                list_result = await session.list_tools()
                
                # Create LangChain tools
                for mcp_tool in list_result.tools:
                    tool = Tool(
                        name=mcp_tool.name,
                        func=create_tool_executor(session, mcp_tool.name),
                        description=mcp_tool.description,
                    )
                    tools.append(tool)
                
                return tools
                
    except Exception as e:
        print(f"Error loading MCP tools: {e}")
        return []

def create_tool_executor(session, tool_name):
    """Create a function that executes an MCP tool"""
    async def async_executor(**kwargs):
        result = await session.call_tool(tool_name, kwargs)
        return result.content[0].text if hasattr(result, 'content') else str(result)
    
    def sync_executor(**kwargs):
        return asyncio.run(async_executor(**kwargs))
    
    return sync_executor

# Usage
async def demo():
    tools = await load_mcp_tools("math_server.py")
    print(f"Loaded {len(tools)} tools")
    
    for tool in tools:
        print(f"Tool: {tool.name}")
        if tool.name == "add":
            result = tool.func(a=10, b=20)
            print(f"10 + 20 = {result}")

if __name__ == "__main__":
    asyncio.run(demo())